# ViT + XAI on 90 × 65 indicator images: Colab / Kaggle runner

Runtime → Change runtime type → **GPU**. The data and results folders live on Google Drive, so a disconnect loses nothing: re-run the cells and finished folds are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Upload the project folder to Drive first (or git clone it here).
PROJECT = '/content/drive/MyDrive/ViT_and_XAI'
%cd $PROJECT

In [ ]:
# TA-Lib >= 0.6 ships Linux wheels that bundle the C library.
!pip -q install "TA-Lib>=0.6.0" yfinance captum timm pyarrow
!pip -q install -e .
import torch, talib; print(torch.__version__, torch.cuda.is_available(), talib.__version__)

In [ ]:
# Optional: tests (about 10 s)
!python -m pytest -q

## Phases 1–4: data, indicators, labels, image check (run once)

In [ ]:
!python scripts/01_download.py
!python scripts/02_indicators.py
!python scripts/03_labels_folds.py
!python scripts/04_check_images.py --fold 0

## Phase 5: training (resumable; finished folds are skipped)
Use `--fold 0 1 2` to train a subset, and `--set train.num_workers=2` for faster loading.

In [ ]:
!python scripts/05_train.py --set train.num_workers=2

## Phases 6–7: evaluation and XAI

In [ ]:
!python scripts/06_evaluate.py
!python scripts/07_xai.py

In [ ]:
!python scripts/08_backtest.py

In [ ]:
from IPython.display import Image, display
for m in ['integrated_gradients', 'row_occlusion', 'chefer']:
    display(Image(f'results/vit_90x65/xai/seed_0/summary/figures/top_indicators_{m}.png'))

## Base-paper replication: 9 ETFs, 65 × 65, θ = 0.01
Separate data folder (`data_paper_etf`) and results (`results/paper_vit_theta001`); the NIFTY 50 run is untouched.

In [ ]:
P = '--config-dir configs/paper_etf'
!python scripts/01_download.py $P
!python scripts/02_indicators.py $P
!python scripts/03_labels_folds.py $P
!python scripts/04_check_images.py $P --fold 0

In [ ]:
!python scripts/05_train.py $P --set train.num_workers=2
!python scripts/06_evaluate.py $P
!python scripts/08_backtest.py $P
!python scripts/07_xai.py $P

## Base-paper replication: balanced θ = 0.0038
Reuses the downloaded data and indicators.

In [ ]:
T = 'labels.theta=0.0038 labels.file=labels_theta0038.parquet run_id=paper_vit_theta0038'
!python scripts/03_labels_folds.py $P --set $T
!python scripts/05_train.py $P --set $T train.num_workers=2
!python scripts/06_evaluate.py $P --set $T
!python scripts/08_backtest.py $P --set $T
!python scripts/07_xai.py $P --set $T

## Base-paper replication: 11-day peak/valley labels (matches the paper's class split)
Without and with class weights. Reuses the downloaded data and indicators.

In [ ]:
V = 'labels.method=peak_valley labels.window=11 labels.file=labels_pv11.parquet'
!python scripts/03_labels_folds.py $P --set $V
for run, extra in [('paper_vit_pv11', ''), ('paper_vit_pv11_cw', 'train.class_weights=true')]:
    !python scripts/05_train.py $P --set $V run_id=$run $extra train.num_workers=2
    !python scripts/06_evaluate.py $P --set $V run_id=$run
    !python scripts/08_backtest.py $P --set $V run_id=$run